# Notebook 3: Experiment 3 — Different Timeframe Training (80/20)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on Weekly/Monthly/Yearly data, predict on Daily test data.  
**Train/Test Split:** 80/20 (chronological, split date from daily data)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Scaler:** ProportionScaler (÷ 10,501)  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [2]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

import plotly.graph_objects as go
from plotly.subplots import make_subplots

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'
EXP_LABEL = f'Exp3_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 3 - Different Timeframe Training (80/20)")


Experiment 3 - Different Timeframe Training (80/20)


In [3]:
# Load ALL timeframe data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)

print("\nLoading weekly data...")
weekly_data = {}
for stock in STOCKS:
    weekly_data[stock] = load_stock_data(get_data_paths(DATA_DIR, stock, 'weekly'))
    print(f"  {stock}: {len(weekly_data[stock])} records")

print("\nLoading monthly data...")
monthly_data = {}
for stock in STOCKS:
    monthly_data[stock] = load_stock_data(get_data_paths(DATA_DIR, stock, 'monthly'))
    print(f"  {stock}: {len(monthly_data[stock])} records")

print("\nLoading yearly data...")
yearly_data = {}
for stock in STOCKS:
    yearly_data[stock] = load_stock_data(get_data_paths(DATA_DIR, stock, 'yearly'))
    print(f"  {stock}: {len(yearly_data[stock])} records")

print("\nAll data loaded!")

timeframe_data = {
    'weekly': weekly_data,
    'monthly': monthly_data,
    'yearly': yearly_data,
}


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

Loading weekly data...
  TLKM: 1102 records
  BBCA: 1102 records
  ASII: 1102 records
  UNVR: 1102 records

Loading monthly data...
  TLKM: 256 records
  BBCA: 256 records
  ASII: 256 records
  UNVR: 256 records

Loading yearly data...
  TLKM: 22 records
  BBCA: 22 records
  ASII: 22 records
  UNVR: 22 records

All data loaded!


In [4]:
# Reload the module to get the latest fixes
import importlib
import stock_prediction_utils
importlib.reload(stock_prediction_utils)
from stock_prediction_utils import *
print("✓ Module reloaded successfully")

stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
✓ Module reloaded successfully


## Run All Timeframe Experiments

In [5]:
# ============================================================
# EXPERIMENT 3: Different timeframe training
# ============================================================
all_results = []

for stock in STOCKS:
    for tf_name in TIMEFRAMES:
        print(f"\n{'#'*60}")
        print(f"# STOCK: {stock} | TRAIN TIMEFRAME: {tf_name}")
        print(f"{'#'*60}")
        
        train_df = timeframe_data[tf_name][stock]
        test_daily_df = daily_data[stock]
        
        # Prepare data
        X_train, y_train, X_test, y_test, test_dates = prepare_diff_timeframe_data(
            train_df, test_daily_df,
            train_ratio=TRAIN_RATIO, lookback=LOOKBACK
        )
        
        if X_train is None:
            print(f"  SKIPPED: Not enough {tf_name} data for {stock}")
            for model_type in MODEL_TYPES:
                all_results.append({
                    'Stock': stock, 'Train_Timeframe': tf_name,
                    'Model': model_type, 'MSE': np.nan, 'RMSE': np.nan,
                    'MAE': np.nan, 'MAPE (%)': np.nan, 'R2': np.nan,
                    'Note': 'Insufficient training data'
                })
            continue
        
        print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
        
        for model_type in MODEL_TYPES:
            exp_name = f'{EXP_LABEL}_{stock}_{tf_name}'
            
            y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
                model_type=model_type,
                X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                experiment_name=exp_name,
                save_dir=f'models/{EXP_LABEL}',
                epochs=EPOCHS, batch_size=BATCH_SIZE
            )
            
            result = {
                'Stock': stock, 'Train_Timeframe': tf_name,
                'Model': model_type, **metrics
            }
            all_results.append(result)
            
            plot_actual_vs_predicted(
                test_dates, y_true_inv, y_pred_inv,
                model_type, f'{stock}_train_{tf_name}',
                EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
            )

print("\n\nAll Experiment 3 (80/20) training complete!")



############################################################
# STOCK: TLKM | TRAIN TIMEFRAME: weekly
############################################################
  X_train: (877, 1, 1), X_test: (1049, 1, 1)

Training BiLSTM for: Exp3_80_20_TLKM_weekly
  Train samples: 877, Test samples: 1049
Epoch 1/100
13/13 [==============================] - ETA: 0s - loss: 0.0163
Epoch 1: val_loss improved from inf to 0.01849, saving model to models/Exp3_80_20\Exp3_80_20_TLKM_weekly_BiLSTM_best.keras
13/13 [==============================] - 13s 142ms/step - loss: 0.0163 - val_loss: 0.0185
Epoch 2/100
13/13 [==============================] - ETA: 0s - loss: 0.0054
Epoch 2: val_loss improved from 0.01849 to 0.00195, saving model to models/Exp3_80_20\Exp3_80_20_TLKM_weekly_BiLSTM_best.keras
13/13 [==============================] - 0s 22ms/step - loss: 0.0054 - val_loss: 0.0019
Epoch 3/100
13/13 [==============================] - ETA: 0s - loss: 0.0042
Epoch 3: val_loss did not improve from 0.00195
13/

## Results Summary

In [6]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 3 - Different Timeframe (80/20)")

results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 3 - Different Timeframe (80/20)
Stock Train_Timeframe  Model          MSE      RMSE       MAE  MAPE (%)        R2  Training_Time_s  Epochs_Run
 TLKM          weekly BiLSTM    3086.8504   55.5594   41.3414    1.3380  0.981931             33.9         100
 TLKM          weekly  BiGRU    3772.6413   61.4218   47.2169    1.5087  0.977917             28.6         100
 TLKM          weekly   LSTM    4018.5556   63.3921   48.7183    1.5788  0.976477             21.0         100
 TLKM          weekly    GRU    3009.7180   54.8609   40.6998    1.3215  0.982382             19.1         100
 TLKM         monthly BiLSTM    4074.4036   63.8311   49.7063    1.5950  0.976150             17.9         100
 TLKM         monthly  BiGRU    3354.3257   57.9165   44.2055    1.4393  0.980365             16.2         100
 TLKM         monthly   LSTM    4126.1802   64.2353   49.3091    1.5734  0.975847             11.5         100
 TLKM         monthly    GRU    3182.8750   56.4170   41.8320    1

## Visualizations

In [7]:
# ============================================================
# INTERACTIVE RESULTS VISUALIZATIONS
# ============================================================

print("Generating interactive results visualizations...\n")

# 1. Interactive Metrics Comparison
print("1. Generating Metrics Comparison Chart...")
fig1, html1 = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2']
)
print(f"   ✓ Saved: {html1}")
fig1.show()

print()

# 2. Model Radar Chart
print("2. Generating Model Radar Chart...")
fig2, html2 = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html2}")
fig2.show()

print()

# 3. Interactive Dashboard with Subplots
print("3. Generating Results Dashboard...")
fig3 = go.Figure()

# RMSE by model
rmse_by_model = results_df.groupby('Model')['RMSE'].mean().sort_values()
fig3.add_trace(go.Bar(
    x=rmse_by_model.index,
    y=rmse_by_model.values,
    name='RMSE',
    marker_color='#0072B2',
    text=np.round(rmse_by_model.values, 4),
    textposition='outside',
    hovertemplate='Model: %{x}<br>Avg RMSE: %{y:.4f}<extra></extra>'
))

fig3.update_layout(
    title=f"<b>{EXP_LABEL} - Average Metrics by Model</b>",
    xaxis_title="Model",
    yaxis_title="RMSE",
    height=600,
    template='plotly_white',
    font=dict(size=12),
    showlegend=False
)

html3 = f'figures/{EXP_LABEL}/{EXP_LABEL}_metrics_dashboard.html'
fig3.write_html(html3)
print(f"   ✓ Saved: {html3}")
fig3.show()

print("\n✓ All interactive results visualizations generated successfully!")

Generating interactive results visualizations...

1. Generating Metrics Comparison Chart...
   ✓ Saved: figures/Exp3_80_20/Exp3_80_20_metrics_comparison.html



2. Generating Model Radar Chart...
   ✓ Saved: figures/Exp3_80_20/Exp3_80_20_model_radar.html



3. Generating Results Dashboard...
   ✓ Saved: figures/Exp3_80_20/Exp3_80_20_metrics_dashboard.html



✓ All interactive results visualizations generated successfully!


## Interactive Results Visualizations

In [8]:
# ============================================================
# METRICS BAR CHARTS PER STOCK
# ============================================================
for stock in STOCKS:
    stock_df = results_df[results_df['Stock'] == stock].copy()
    if stock_df.empty:
        continue
    
    for metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:
        fig, ax = plt.subplots(figsize=(12, 6))
        
        timeframes = stock_df['Train_Timeframe'].unique()
        n_tf = len(timeframes)
        n_models = len(MODEL_TYPES)
        bar_width = 0.8 / n_models
        x = np.arange(n_tf)
        
        for i, model_type in enumerate(MODEL_TYPES):
            vals = []
            for tf in timeframes:
                v = stock_df[(stock_df['Model'] == model_type) & 
                             (stock_df['Train_Timeframe'] == tf)][metric]
                vals.append(v.values[0] if len(v) > 0 and not pd.isna(v.values[0]) else 0)
            
            ax.bar(x + i * bar_width, vals, bar_width,
                   label=model_type, color=MODEL_COLORS[model_type],
                   edgecolor='white', linewidth=0.5)
        
        ax.set_xlabel('Training Timeframe', fontsize=20)
        ax.set_ylabel(metric, fontsize=20)
        ax.set_title(f'{EXP_LABEL} | {stock} - {metric} by Timeframe', fontsize=20)
        ax.set_xticks(x + bar_width * (n_models - 1) / 2)
        ax.set_xticklabels(timeframes, fontsize=20)
        ax.legend(fontsize=20)
        fig.tight_layout()
        
        fname = f'figures/{EXP_LABEL}/{stock}_{metric}_by_timeframe.png'.replace('(%)', 'pct')
        save_fig(fig, fname)

print("All bar charts saved!")


  Figure saved: figures/Exp3_80_20/TLKM_RMSE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/TLKM_MAE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/TLKM_MAPE pct_by_timeframe.png
  Figure saved: figures/Exp3_80_20/TLKM_R2_by_timeframe.png
  Figure saved: figures/Exp3_80_20/BBCA_RMSE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/BBCA_MAE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/BBCA_MAPE pct_by_timeframe.png
  Figure saved: figures/Exp3_80_20/BBCA_R2_by_timeframe.png
  Figure saved: figures/Exp3_80_20/ASII_RMSE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/ASII_MAE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/ASII_MAPE pct_by_timeframe.png
  Figure saved: figures/Exp3_80_20/ASII_R2_by_timeframe.png
  Figure saved: figures/Exp3_80_20/UNVR_RMSE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/UNVR_MAE_by_timeframe.png
  Figure saved: figures/Exp3_80_20/UNVR_MAPE pct_by_timeframe.png
  Figure saved: figures/Exp3_80_20/UNVR_R2_by_timeframe.png
All 

In [9]:
# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("  BEST TIMEFRAME PER STOCK (by RMSE)")
print("="*70)
for stock in STOCKS:
    stock_data = results_df[results_df['Stock'] == stock].dropna(subset=['RMSE'])
    if stock_data.empty:
        continue
    best_idx = stock_data['RMSE'].idxmin()
    best = stock_data.loc[best_idx]
    print(f"  {stock}: {best['Train_Timeframe']} + {best['Model']} "
          f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")



  BEST TIMEFRAME PER STOCK (by RMSE)
  TLKM: weekly + GRU (RMSE=54.8609, R²=0.982382)
  BBCA: weekly + BiGRU (RMSE=134.0270, R²=0.983380)
  ASII: monthly + BiGRU (RMSE=82.5500, R²=0.981555)
  UNVR: weekly + LSTM (RMSE=69.1997, R²=0.994495)


In [ ]:
# ============================================================
# INTERACTIVE VISUALIZATION - ACTUAL VS PREDICTED (Plotly)
# ============================================================

print("\n" + "="*70)
print("GENERATING INTERACTIVE VISUALIZATIONS")
print("="*70 + "\n")

import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("1. Generating interactive prediction plots for each stock...\n")

for stock in STOCKS:
    print(f"   Generating interactive plot for {stock}...")
    
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    
    # Create subplots (one for each model)
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=MODEL_TYPES,
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]],
        vertical_spacing=0.12,
        horizontal_spacing=0.1
    )
    
    row_col_pairs = [(1, 1), (1, 2), (2, 1), (2, 2)]
    
    for idx, model_type in enumerate(MODEL_TYPES):
        row, col = row_col_pairs[idx]
        y_pred = all_predictions[stock][model_type][1]
        
        # Actual prices
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_true,
                name='Actual Price',
                mode='lines',
                line=dict(color='#9A9A9A', width=2),
                hovertemplate='<b>Actual Price</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
                showlegend=(idx == 0),
                legendgroup="actual"
            ),
            row=row, col=col
        )
        
        # Predicted prices
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_pred,
                name=f'{model_type} Prediction',
                mode='lines',
                line=dict(color=MODEL_COLORS[model_type], width=2, dash='dash'),
                hovertemplate='<b>' + model_type + ' Prediction</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
                showlegend=(idx == 0),
                legendgroup=model_type
            ),
            row=row, col=col
        )
        
        # Update axes labels
        fig.update_xaxes(title_text="Date", row=row, col=col, tickfont=dict(size=14, family="Times New Roman"))
        fig.update_yaxes(title_text="Price (IDR)", row=row, col=col, tickfont=dict(size=14, family="Times New Roman"))
    
    # Update layout
    fig.update_layout(
        title=f'<b>{stock} - Actual vs Predicted Prices ({EXP_LABEL})</b>',
        height=900,
        template='plotly_white',
        hovermode='x unified',
        showlegend=True,
        font=dict(size=20, family="Times New Roman"),
        legend=dict(
            x=1.05,
            y=0.99,
            xanchor="left",
            yanchor="top",
        )
    )
    
    # Save as HTML
    html_filename = f'figures/{EXP_LABEL}/interactive_{stock}_all_models.html'
    fig.write_html(html_filename)
    print(f"      ✓ Saved: {html_filename}")
    fig.show()

print("\n" + "="*70)
print("2. Generating interactive toggle plots with buttons...\n")

for stock in STOCKS:
    print(f"   Generating toggle plot for {stock}...")
    
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    
    fig = go.Figure()
    
    # Add actual price (always visible)
    fig.add_trace(
        go.Scatter(
            x=dates, y=y_true,
            name='Actual Price',
            mode='lines',
            line=dict(color='#9A9A9A', width=2.5),
            hovertemplate='<b>Actual Price</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
            visible=True
        )
    )
    
    # Add all model predictions
    for model_type in MODEL_TYPES:
        y_pred = all_predictions[stock][model_type][1]
        
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_pred,
                name=model_type,
                mode='lines',
                line=dict(color=MODEL_COLORS[model_type], width=2),
                hovertemplate='<b>' + model_type + '</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
                visible=False
            )
        )
    
    # Create buttons for toggling models
    buttons = [
        dict(
            label='All Models',
            method='update',
            args=[{'visible': [True] + [False]*len(MODEL_TYPES)},
                  {'title': f'<b>{stock} - All Models ({EXP_LABEL})</b>'}]
        ),
        dict(
            label='Actual Only',
            method='update',
            args=[{'visible': [True] + [False]*len(MODEL_TYPES)},
                  {'title': f'<b>{stock} - Actual Price ({EXP_LABEL})</b>'}]
        )
    ]
    
    for idx, model_type in enumerate(MODEL_TYPES):
        visible = [True] + [False]*(len(MODEL_TYPES))
        visible[idx + 1] = True
        buttons.append(
            dict(
                label=model_type,
                method='update',
                args=[{'visible': visible},
                      {'title': f'<b>{stock} - {model_type} Prediction ({EXP_LABEL})</b>'}]
            )
        )
    
    fig.update_layout(
        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.11,
                xanchor="left",
                y=1.15,
                yanchor="top"
            )
        ],
        title=f'<b>{stock} - Toggle Models ({EXP_LABEL})</b>',
        xaxis_title='Date',
        yaxis_title='Close Price (IDR)',
        hovermode='x unified',
        template='plotly_white',
        font=dict(size=20, family="Times New Roman"),
        height=600,
        margin=dict(l=50, r=50, t=120, b=50)
    )
    
    html_filename = f'figures/{EXP_LABEL}/interactive_{stock}_toggle.html'
    fig.write_html(html_filename)
    print(f"      ✓ Saved: {html_filename}")
    fig.show()

print("\n" + "="*70)
print("3. Generating interactive results visualizations...\n")

# 1. Interactive Results Dashboard
print("   Generating Results Dashboard...")
fig1, html1 = create_interactive_results_dashboard_exp1(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"      ✓ Saved: {html1}")
fig1.show()

print()

# 2. Interactive Heatmaps for each metric
print("   Generating Interactive Heatmaps...")
for metric in ['RMSE', 'MAE', 'R2', 'MAPE (%)']:
    try:
        fig, html = create_interactive_metrics_heatmap_exp1(
            results_df, metric, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )
        print(f"      ✓ {metric} heatmap: {html}")
        fig.show()
    except Exception as e:
        print(f"      ⚠ Skipping {metric}: {str(e)}")

print()

# 3. Metrics Comparison Chart
print("   Generating Metrics Comparison Chart...")
fig3, html3 = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2']
)
print(f"      ✓ Saved: {html3}")
fig3.show()

print()

# 4. Model Radar Chart
print("   Generating Model Radar Chart...")
fig4, html4 = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"      ✓ Saved: {html4}")
fig4.show()

print("\n" + "="*70)
print("✓ ALL INTERACTIVE VISUALIZATIONS GENERATED SUCCESSFULLY!")
print("="*70)

: 

: 

: 

## Interactive Prediction Visualizations (Plotly)
Explore interactive charts showing actual vs predicted prices with zoom, pan, and hover details. Toggle between models, compare results, and view performance metrics.